# NLI fine-tuning v2 — deterministic build

XLM-R-large-XNLI fine-tuned on the implicit citation benchmark.
This version has reproducible 5-fold OOF: identical metrics across runs (same seed, fp32, deterministic cuDNN).


In [ ]:
# ============================================================
# DETERMINISM SETUP — RUN THIS CELL FIRST, right after
# "Runtime > Restart session". CUBLAS_WORKSPACE_CONFIG and
# torch.use_deterministic_algorithms need a FRESH CUDA context:
# if any other cell imported torch / touched the GPU before this
# one, the CUBLAS lock is silently ignored. So: restart runtime,
# then run THIS cell before anything else.
# ============================================================
import os
os.environ["PYTHONHASHSEED"]          = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # required for deterministic cuBLAS GEMM

# True  -> real lock: raises if an op has no deterministic kernel.
# False -> soft (warn_only): runs but residual non-determinism is possible.
# If you hit "... does not have a deterministic implementation ...",
# flip this to False (that op is then the only remaining noise source).
STRICT_DETERMINISM = True

import torch
torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False
torch.backends.cuda.matmul.allow_tf32 = False   # <-- lock added: no TF32 (A100 enables it by default)
torch.backends.cudnn.allow_tf32       = False   # <-- lock added
torch.use_deterministic_algorithms(True, warn_only=not STRICT_DETERMINISM)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Determinism locked | strict={STRICT_DETERMINISM} "
      f"| TF32={torch.backends.cuda.matmul.allow_tf32} "
      f"| cudnn.deterministic={torch.backends.cudnn.deterministic}")


In [ ]:
import os, re, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef,
                              balanced_accuracy_score, precision_score, recall_score)
from scipy.stats import fisher_exact
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:

BASE_PATH = '.'  # repo root
DATA_PATH = 'DATA/outputs/benchmark.csv'
OUTPUT_PATH = 'DATA/outputs/predictions/baselines'  # OOF .npy shipped here
os.makedirs(OUTPUT_PATH, exist_ok=True)


## Config + reproducibility helpers


In [ ]:
SEED = 42
N_SPLITS = 5

MODEL_NAME = 'joeddav/xlm-roberta-large-xnli'
MAX_LEN = 512

# Training — conservative, stable hyperparameters
BATCH_SIZE = 16          # A100
GRAD_ACCUM = 1
LR = 1e-5
N_EPOCHS = 4
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Output filename for the .npy of OOF probabilities (positive class)
NPY_NAME = 'xlm-r-xnli.npy'   # consistent with ST-MiniLM.npy etc.

def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Re-assert deterministic flags (defensive: in case something flipped them)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32       = False

set_all_seeds(SEED)


def _seed_worker(worker_id):
    # For DataLoader workers if num_workers > 0 — harmless if num_workers=0.
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


## Data loading + label resolution


In [ ]:
df = pd.read_csv(DATA_PATH)

def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r'\boui\b', s) is not None
    has_non = re.search(r'\bnon\b', s) is not None
    if has_oui and not has_non: return 'oui'
    if has_non and not has_oui: return 'non'
    return np.nan

df['a1'] = df['eval_A1'].apply(extract_oui_non)
df['a2'] = df['eval_A2'].apply(extract_oui_non)
df['a3'] = df['eval_A3'].apply(extract_oui_non)

def resolve_label(r):
    a1, a2, a3 = r['a1'], r['a2'], r['a3']
    if pd.notna(a1) and pd.notna(a2) and a1 == a2:
        return a1
    if pd.notna(a1) and pd.notna(a2) and a1 != a2 and pd.notna(a3):
        return a3
    return np.nan

df['label_str'] = df.apply(resolve_label, axis=1)
df = df[df['label_str'].isin(['oui', 'non'])].copy().reset_index(drop=True)
df['label'] = df['label_str'].map({'oui': 1, 'non': 0}).astype(int)
df['agreement'] = (df['a1'] == df['a2']).astype(int)

for c in ['text', 'article_text', 'pred_art']:
    df[c] = df[c].fillna('').astype(str)

print(f"Total: {len(df)} | YES: {(df['label']==1).sum()} | NO: {(df['label']==0).sum()}")
print(f"Agreement: {(df['agreement']==1).sum()} | Disagreement: {(df['agreement']==0).sum()}")


In [ ]:
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby('decision_id').size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out['fold'] = out['decision_id'].map(group_to_fold).astype(int)
    return out, fold_loads

df, fold_loads = make_grouped_folds(df, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")
for f in range(N_SPLITS):
    sub = df[df['fold']==f]
    print(f"  Fold {f}: n={len(sub)}, YES={sub['label'].mean():.1%}, disagreement={1-sub['agreement'].mean():.1%}")


## Dataset + training


In [ ]:
class NLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.premises = df['text'].tolist()
        self.hypotheses = [
            f"Ce passage applique la règle suivante : {art}"
            for art in df['article_text'].tolist()
        ]
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.premises[idx],
            self.hypotheses[idx],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


## Train one fold
Reset seed at start of every fold so fold k is independent of folds 0..k-1.


In [ ]:
def train_one_fold(df_train, df_test, tokenizer, fold_id):
    """Train on df_train, return OOF probabilities (positive class) on df_test."""

    # Re-seed at the start of every fold for reproducibility.
    fold_seed = SEED + fold_id
    set_all_seeds(fold_seed)

    # XLM-R-large NLI fine-tune, swap the 3-class head for a 2-class head
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        ignore_mismatched_sizes=True,
        torch_dtype=torch.float32  # explicit — no bf16 (determinism)
    ).to(device)

    nan_params = [n for n, p in model.named_parameters() if torch.isnan(p).any()]
    if nan_params:
        print(f"⚠️  NaN detected in initial weights: {nan_params[:3]}...")
        raise ValueError("Model loaded with NaN weights, abort.")

    train_ds = NLIDataset(df_train, tokenizer, MAX_LEN)
    test_ds  = NLIDataset(df_test,  tokenizer, MAX_LEN)

    # Explicit generator pinned to fold_seed for reproducible shuffling.
    g = torch.Generator()
    g.manual_seed(fold_seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=g, worker_init_fn=_seed_worker,
    )
    test_loader = DataLoader(
        test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * N_EPOCHS // GRAD_ACCUM
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps
    )

    # Training
    model.train()
    for epoch in range(N_EPOCHS):
        epoch_loss = 0
        n_valid_steps = 0
        n_skipped = 0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f'Fold {fold_id} Epoch {epoch+1}/{N_EPOCHS}')

        for step, batch in enumerate(pbar):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                labels=batch['label']
            )
            loss = outputs.loss / GRAD_ACCUM

            if torch.isnan(loss) or torch.isinf(loss):
                n_skipped += 1
                optimizer.zero_grad()
                continue

            loss.backward()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            epoch_loss += loss.item() * GRAD_ACCUM
            n_valid_steps += 1
            avg_loss = epoch_loss / max(n_valid_steps, 1)
            pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'skipped': n_skipped})

        if n_skipped > 0:
            print(f"  ⚠️  Epoch {epoch+1}: {n_skipped} batches skipped (NaN loss)")

    # Eval
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Fold {fold_id} eval'):
            batch_inputs = {
                'input_ids':      batch['input_ids'].to(device),
                'attention_mask': batch['attention_mask'].to(device)
            }
            logits = model(**batch_inputs).logits.float()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs.tolist())

    del model
    torch.cuda.empty_cache()

    return np.array(all_probs)


In [ ]:
# Sanity check on fold 0 — verify loss decreases and probs calibrate
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("=" * 60)
print("TEST FOLD 0 — check that the loss decreases")
print("=" * 60)
df_train_0 = df[df['fold'] != 0].reset_index(drop=True)
df_test_0  = df[df['fold'] == 0].copy()
test_idx_0 = df[df['fold'] == 0].index

probs_0 = train_one_fold(df_train_0, df_test_0, tokenizer, 0)

y_test = df.loc[test_idx_0, 'label'].values
print(f"\n=== Sanity check fold 0 ===")
print(f"Probs — min: {probs_0.min():.3f}, max: {probs_0.max():.3f}, mean: {probs_0.mean():.3f}")
print(f"Probs on YES gold: mean={probs_0[y_test==1].mean():.3f}")
print(f"Probs on NO gold:  mean={probs_0[y_test==0].mean():.3f}")
print(f"If mean(YES) > mean(NO), the model learned something ✓")


In [ ]:
# Full cross-validation — folds 1..4 (fold 0 already computed above)
p_oof = np.zeros(len(df), dtype=np.float64)
p_oof[test_idx_0] = probs_0

for fold_id in range(1, N_SPLITS):
    print(f"\n{'='*60}\nFOLD {fold_id+1}/{N_SPLITS}\n{'='*60}")
    df_train = df[df['fold'] != fold_id].reset_index(drop=True)
    df_test  = df[df['fold'] == fold_id].copy()
    test_idx = df[df['fold'] == fold_id].index

    print(f"Train: {len(df_train)} | Test: {len(df_test)}")
    probs = train_one_fold(df_train, df_test, tokenizer, fold_id)
    p_oof[test_idx] = probs

    # Save after each fold (safety)
    np.save(f'{OUTPUT_PATH}/p_oof_partial.npy', p_oof)

df['nli_p_oui'] = p_oof
df.to_csv(f'{OUTPUT_PATH}/nli_finetuned_results.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_PATH}/nli_finetuned_results.csv")

# Save OOF probs as .npy (1D, order = df order)
# Same format as ST-MiniLM.npy etc.: np.float64, shape (n,)
npy_path = f'{OUTPUT_PATH}/{NPY_NAME}'
np.save(npy_path, p_oof.astype(np.float64))
print(f"✓ Saved probs .npy: {npy_path}")
print(f"  shape={p_oof.shape}, dtype=float64, "
      f"min={p_oof.min():.3f}, max={p_oof.max():.3f}, mean={p_oof.mean():.3f}")


## Threshold tuning + metrics


In [ ]:
y_true = df['label'].values
p_oui  = df['nli_p_oui'].values

def find_best_threshold(y_true, p_oui, metric='MCC'):
    best_thr, best_val = 0.5, -1e18
    for thr in np.arange(0.05, 0.96, 0.01):
        y_pred = (p_oui >= thr).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        if metric == 'MCC':
            val = matthews_corrcoef(y_true, y_pred)
        elif metric == 'F1':
            val = f1_score(y_true, y_pred, pos_label=1)
        if val > best_val:
            best_val, best_thr = val, thr
    return best_thr, best_val

best_thr, _ = find_best_threshold(y_true, p_oui, 'MCC')
y_pred = (p_oui >= best_thr).astype(int)

print(f"Optimal threshold (MCC): {best_thr:.2f}")
print(f"\n=== NLI fine-tuned metrics (XLM-R-large-XNLI) ===")
print(f"  Accuracy    : {accuracy_score(y_true, y_pred):.3f}")
print(f"  Bal. Acc.   : {balanced_accuracy_score(y_true, y_pred):.3f}")
print(f"  F1 (pos)    : {f1_score(y_true, y_pred, pos_label=1):.3f}")
print(f"  Precision   : {precision_score(y_true, y_pred, pos_label=1):.3f}")
print(f"  Recall      : {recall_score(y_true, y_pred, pos_label=1):.3f}")
print(f"  MCC         : {matthews_corrcoef(y_true, y_pred):.3f}")

print(f"\n=== Paper comparison ===")
print(f"  Supervised ensemble: F1=0.70, Acc=0.77, MCC=0.53")
print(f"  SAUL-7B            : F1=0.69, Acc=0.74, MCC=0.47")
print(f"  TF-IDF             : F1=0.63, Acc=0.71, MCC=0.41")


## FP concentration on annotator-disagreement cases (Wald on log OR)


In [ ]:
# Concentration FP agree/disagree — XLM-R-large-XNLI fine-tuned
# Wald on log(OR) via Table2x2 (consistent with Table 11 of the paper)
from statsmodels.stats.contingency_tables import Table2x2

neg = df[df['label'] == 0].copy()
neg['nli_pred'] = (neg['nli_p_oui'].values >= best_thr).astype(int)

agree_neg    = neg[neg['agreement'] == 1]
disagree_neg = neg[neg['agreement'] == 0]

fp_agree    = int((agree_neg['nli_pred'] == 1).sum())
fp_disagree = int((disagree_neg['nli_pred'] == 1).sum())
n_agree     = len(agree_neg)
n_disagree  = len(disagree_neg)

fpr_agree    = fp_agree    / n_agree    if n_agree    else 0.0
fpr_disagree = fp_disagree / n_disagree if n_disagree else 0.0

# Layout: rows = [disagree, agree], cols = [FP, not-FP]
table = [[fp_disagree, n_disagree - fp_disagree],
         [fp_agree,    n_agree    - fp_agree]]
t22 = Table2x2(table)
OR     = t22.oddsratio
p_wald = t22.oddsratio_pvalue()
ci_lo, ci_hi = np.exp(t22.log_oddsratio_confint(alpha=0.05))

total_fp           = fp_agree + fp_disagree
pct_fp_on_disagree = fp_disagree / total_fp if total_fp > 0 else 0.0

print("=" * 60)
print("ERROR CONCENTRATION — XLM-R-large-XNLI fine-tuned")
print("=" * 60)
print(f"\nTrue negatives on agreement  (n={n_agree}): FP={fp_agree}, FPR={fpr_agree:.1%}")
print(f"True negatives on disagreement (n={n_disagree}): FP={fp_disagree}, FPR={fpr_disagree:.1%}")
print(f"\nOdds Ratio: {OR:.2f}  (95% CI [{ci_lo:.2f}, {ci_hi:.2f}])")
print(f"p-value (Wald on log(OR), two-sided): {p_wald:.4f}")
print(f"\n{fp_disagree}/{total_fp} = {pct_fp_on_disagree:.1%} of FP fall on the "
      f"{n_disagree/(n_agree+n_disagree):.0%} of disagreement cases")

print(f"\n=== Paper comparison (Table 11, same methodology) ===")
print(f"  Supervised ensemble: OR=1.83, p=.030, 68% FP on 33% disagreement")
print(f"  OR range, other models: 1.28 to 2.91")


In [ ]:
# Final save (uses OR / p_wald, not odds / p_val)
summary = pd.DataFrame([{
    'Model':              'XLM-R-large-XNLI (fine-tuned)',
    'Threshold':          round(best_thr, 2),
    'Accuracy':           round(accuracy_score(y_true, y_pred), 3),
    'F1_pos':             round(f1_score(y_true, y_pred, pos_label=1), 3),
    'MCC':                round(matthews_corrcoef(y_true, y_pred), 3),
    'FPR_agree':          f'{fpr_agree:.1%}',
    'FPR_disagree':       f'{fpr_disagree:.1%}',
    'OR':                 round(OR, 2),
    'CI95_low':           round(ci_lo, 2),
    'CI95_high':          round(ci_hi, 2),
    'p_value':            round(p_wald, 4),
    'pct_FP_on_disagree': f'{pct_fp_on_disagree:.1%}',
}])

print(summary.to_string(index=False))
summary.to_csv(f'{OUTPUT_PATH}/nli_finetuned_summary.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_PATH}/nli_finetuned_summary.csv")
print(f"✓ .npy already written to: {OUTPUT_PATH}/{NPY_NAME}")
